<a href="https://colab.research.google.com/github/hamza-24-ai/Pytorch_with_practical_deepLearning/blob/main/CNN_07_Mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

In [2]:
torch.manual_seed(42)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"your device is runing on {device}")

your device is runing on cuda


In [4]:
df = pd.read_csv("/mnist_train.csv")
df.head()

,label,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
df.shape

(60000, 785)

In [6]:
# scale the dataset

X = df.iloc[:,1:].values
y = df.iloc[:,0].values


In [7]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [8]:
X_train = X_train/255.0
X_test = X_test/255.0

In [10]:
class CustomDataSet(Dataset):

  def __init__(self, features, labels):
    # self.features = torch.tensor(features, dtype=torch.float32) to get 1D data
    # but we need 2D dataset
    self.features = torch.tensor(features, dtype = torch.float32).reshape(-1,1,28,28)
    # -1 btata hai no idea kitna batches aee ga jbka 1 btata hai grayscale hai or 28x28 channel hai
    self.labels = torch.tensor(labels, dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]



In [11]:
train_dataset = CustomDataSet(X_train, y_train)
print("Features have been trained successfully")

Features have been trained successfully


In [12]:
test_dataset = CustomDataSet(X_test, y_test)
print("Testing custom data have been generated")

Testing custom data have been generated


In [13]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)
print("Loader have been done successfully")

Loader have been done successfully


In [17]:
class MyNN(nn.Module):
  def __init__(self, input_features):
    # mtlb ka kitna channel ki image input features sa muraad

    super().__init__()

    # Make image features & classification by convolutional Operation and Pooling
    self.features = nn.Sequential(
        nn.Conv2d(input_features,32,kernel_size=3, padding="same"),
        nn.ReLU(),
        nn.BatchNorm2d(32),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(32,64,kernel_size=3, padding="same"),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(kernel_size=2, stride=2),
    )
    self.classifier = nn.Sequential(
        nn.Flatten(),
        nn.Linear(64*7*7, 128),
        nn.ReLU(),
        nn.Dropout(p=0.4),

        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Dropout(p=0.4),

        nn.Linear(64,10)
    )


  def forward(self,x):

    x = self.features(x)
    x = self.classifier(x)

    return x



In [18]:
learning_rate = 0.01
epochs = 100

In [21]:
model = MyNN(1)

model.to(device)

criterian = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

In [23]:
# training epochs

for epoch in range(epochs):

  total_epoch_loss = 0

  for batch_feature, batch_labels in train_loader:

    # move data to gpu
    batch_feature, batch_labels = batch_feature.to(device), batch_labels.to(device)

    # forward pass
    outputs = model(batch_feature)

    # calculate loss

    loss = criterian(outputs, batch_labels)

    # back pass
    optimizer.zero_grad()
    loss.backward()

    # update grads
    optimizer.step()

    total_epoch_loss = total_epoch_loss + loss.item()

  avg_loss = total_epoch_loss/len(train_loader)
  print(f"Epoch loss : {avg_loss} on epoch No : {epoch+1}")

Epoch loss : 0.19303974510356783 on epoch No : 1
Epoch loss : 0.09848951244788866 on epoch No : 2
Epoch loss : 0.07167644066115221 on epoch No : 3
Epoch loss : 0.05684063888993114 on epoch No : 4
Epoch loss : 0.048132968217115074 on epoch No : 5
Epoch loss : 0.03969633389634934 on epoch No : 6
Epoch loss : 0.03339693605612653 on epoch No : 7
Epoch loss : 0.029360032614747372 on epoch No : 8
Epoch loss : 0.02522204350561757 on epoch No : 9
Epoch loss : 0.023867618623931774 on epoch No : 10
Epoch loss : 0.02154755304804227 on epoch No : 11
Epoch loss : 0.018895453693383996 on epoch No : 12
Epoch loss : 0.017092779171111638 on epoch No : 13
Epoch loss : 0.01414570533647202 on epoch No : 14
Epoch loss : 0.013137592468279764 on epoch No : 15
Epoch loss : 0.011757395001775877 on epoch No : 16
Epoch loss : 0.010479346259498016 on epoch No : 17
Epoch loss : 0.010605176892731227 on epoch No : 18
Epoch loss : 0.009127296416024668 on epoch No : 19
Epoch loss : 0.009901845766435751 on epoch No : 2

In [24]:
model.eval()

MyNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [25]:
# Evaluation dataset

total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in test_loader:

    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)
    total = total + batch_labels.shape[0]
    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)




0.9918333333333333


In [26]:
# Evaluation dataset

total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in train_loader:

    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)
    total = total + batch_labels.shape[0]
    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)




1.0
